# Execution Algorithms

This notebook demonstrates common execution algorithms used in electronic trading:
- **TWAP** (Time-Weighted Average Price): Uniform execution over time
- **VWAP** (Volume-Weighted Average Price): Execution based on volume patterns
- **POV** (Percentage of Volume): Participation-based execution

We'll compare these algorithms and analyze their performance characteristics.

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import custom modules
from utils import generate_price_series, generate_trade_data
from execution_algos import (
    TWAPAlgorithm,
    VWAPAlgorithm,
    POVAlgorithm,
    compare_algorithms,
    calculate_algorithm_performance
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✓ Libraries loaded successfully')

## 1. Generate Synthetic Market Data

Create realistic market data with prices, volumes, and intraday patterns.

In [ ]:
# Generate price series
np.random.seed(42)
n_periods = 390  # One trading day (minutes)
initial_price = 100.0

prices = generate_price_series(
    n_periods=n_periods,
    initial_price=initial_price,
    mu=0.00005,
    sigma=0.01,
    seed=42
)

# Generate trade data with volume
trade_data = generate_trade_data(
    prices=prices,
    avg_volume=5000,
    volume_std=2000,
    seed=42
)

# Create intraday volume profile (U-shaped)
time_idx = np.arange(len(trade_data))
morning_boost = np.exp(-((time_idx - 0) / 50) ** 2)
afternoon_boost = np.exp(-((time_idx - n_periods) / 50) ** 2)
volume_profile = 1 + 2 * (morning_boost + afternoon_boost)
trade_data['volume'] = (trade_data['volume'] * volume_profile).astype(int)

# Combine into market data
market_data = pd.DataFrame({
    'price': prices,
    'volume': trade_data['volume']
})

print(f'Generated {len(market_data)} periods of market data')
print(f'Price range: ${market_data["price"].min():.2f} - ${market_data["price"].max():.2f}')
print(f'Average volume: {market_data["volume"].mean():,.0f} shares')
print(f'Total volume: {market_data["volume"].sum():,.0f} shares')

## 2. Visualize Market Data

Examine the price movements and volume patterns.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot prices
axes[0].plot(market_data.index, market_data['price'], linewidth=1.5, color='blue')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Intraday Price Movement')
axes[0].grid(True, alpha=0.3)

# Plot volume
axes[1].bar(market_data.index, market_data['volume'], 
            width=pd.Timedelta(minutes=0.8), alpha=0.6, color='green')
axes[1].set_ylabel('Volume (shares)')
axes[1].set_xlabel('Time')
axes[1].set_title('Intraday Volume Profile (U-shaped pattern)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('✓ Market data visualization complete')

## 3. TWAP Algorithm

Time-Weighted Average Price algorithm splits the order into equal slices and executes uniformly over time.

In [ ]:
# Define execution parameters
total_quantity = 100000  # shares to execute
start_time = market_data.index[0]
end_time = market_data.index[-1]

# Create TWAP algorithm
twap = TWAPAlgorithm(
    total_quantity=total_quantity,
    start_time=start_time,
    end_time=end_time,
    num_slices=20  # Execute in 20 equal slices
)

# Generate execution schedule
twap_schedule = twap.generate_schedule()

print('TWAP Execution Schedule:')
print('=' * 60)
print(f'Total quantity: {total_quantity:,} shares')
print(f'Number of slices: {len(twap_schedule)}')
print(f'Shares per slice: {twap_schedule["quantity"].mean():,.0f}')
print('\nFirst 5 executions:')
print(twap_schedule.head())

In [ ]:
# Execute TWAP
twap_executions = twap.execute(market_data)

# Calculate performance metrics
arrival_price = market_data['price'].iloc[0]
twap_performance = calculate_algorithm_performance(
    twap_executions,
    benchmark_price=arrival_price,
    side='buy'
)

print('\nTWAP Performance:')
print('=' * 60)
print(f'Arrival price: ${arrival_price:.2f}')
print(f'Average execution price: ${twap_performance["avg_execution_price"]:.2f}')
print(f'Cost vs arrival: {twap_performance["cost_vs_benchmark_bps"]:.2f} bps')
print(f'Total executed: {twap_performance["total_quantity"]:,.0f} shares')
print(f'Number of executions: {twap_performance["num_executions"]}')

## 4. VWAP Algorithm

Volume-Weighted Average Price algorithm executes in proportion to historical volume patterns.

In [ ]:
# Create VWAP algorithm with volume profile
vwap = VWAPAlgorithm(
    total_quantity=total_quantity,
    start_time=start_time,
    end_time=end_time,
    volume_profile=market_data['volume']
)

# Execute VWAP
vwap_executions = vwap.execute(market_data)

# Calculate performance
vwap_performance = calculate_algorithm_performance(
    vwap_executions,
    benchmark_price=arrival_price,
    side='buy'
)

print('VWAP Performance:')
print('=' * 60)
print(f'Average execution price: ${vwap_performance["avg_execution_price"]:.2f}')
print(f'Cost vs arrival: {vwap_performance["cost_vs_benchmark_bps"]:.2f} bps')
print(f'Total executed: {vwap_performance["total_quantity"]:,.0f} shares')
print(f'Number of executions: {vwap_performance["num_executions"]}')
print('\nExecution follows volume profile - more shares during high volume periods')

## 5. POV Algorithm

Percentage of Volume algorithm executes as a target percentage of market volume.

In [ ]:
# Create POV algorithm
pov = POVAlgorithm(
    total_quantity=total_quantity,
    start_time=start_time,
    end_time=end_time,
    target_pov=0.15,  # Target 15% of market volume
    min_quantity=100
)

# Execute POV
pov_executions = pov.execute(market_data)

# Calculate performance
pov_performance = calculate_algorithm_performance(
    pov_executions,
    benchmark_price=arrival_price,
    side='buy'
)

print('POV Performance:')
print('=' * 60)
print(f'Target participation rate: 15%')
print(f'Average execution price: ${pov_performance["avg_execution_price"]:.2f}')
print(f'Cost vs arrival: {pov_performance["cost_vs_benchmark_bps"]:.2f} bps')
print(f'Total executed: {pov_performance["total_quantity"]:,.0f} shares')
print(f'Number of executions: {pov_performance["num_executions"]}')
if 'participation_rate' in pov_executions.columns:
    print(f'Average participation rate: {pov_executions["participation_rate"].mean():.2%}')

## 6. Visualize Execution Schedules

Compare how each algorithm distributes trades over time.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# TWAP execution
axes[0].bar(twap_executions['timestamp'], twap_executions['quantity'], 
            width=pd.Timedelta(minutes=5), alpha=0.7, color='blue', label='TWAP')
axes[0].set_ylabel('Quantity (shares)')
axes[0].set_title('TWAP Execution Schedule - Uniform Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# VWAP execution
axes[1].bar(vwap_executions['timestamp'], vwap_executions['quantity'], 
            width=pd.Timedelta(minutes=0.8), alpha=0.7, color='green', label='VWAP')
axes[1].plot(market_data.index, market_data['volume'] / 10, 
             linewidth=2, color='black', alpha=0.5, label='Market Volume (scaled)')
axes[1].set_ylabel('Quantity (shares)')
axes[1].set_title('VWAP Execution Schedule - Follows Volume Profile')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# POV execution
axes[2].bar(pov_executions['timestamp'], pov_executions['quantity'], 
            width=pd.Timedelta(minutes=0.8), alpha=0.7, color='red', label='POV')
axes[2].plot(market_data.index, market_data['volume'] / 10, 
             linewidth=2, color='black', alpha=0.5, label='Market Volume (scaled)')
axes[2].set_ylabel('Quantity (shares)')
axes[2].set_xlabel('Time')
axes[2].set_title('POV Execution Schedule - Target 15% Participation')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('✓ Execution schedule visualization complete')

## 7. Visualize Execution Progress

Show cumulative execution over time for each algorithm.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Cumulative execution
twap_cumulative = twap_executions['quantity'].cumsum()
vwap_cumulative = vwap_executions['quantity'].cumsum()
pov_cumulative = pov_executions['quantity'].cumsum()

axes[0].plot(twap_executions['timestamp'], twap_cumulative, 
             linewidth=2, label='TWAP', marker='o', markersize=4)
axes[0].plot(vwap_executions['timestamp'], vwap_cumulative, 
             linewidth=2, label='VWAP', marker='s', markersize=3)
axes[0].plot(pov_executions['timestamp'], pov_cumulative, 
             linewidth=2, label='POV', marker='^', markersize=3)
axes[0].axhline(total_quantity, color='black', linestyle='--', alpha=0.5, label='Target')
axes[0].set_ylabel('Cumulative Shares Executed')
axes[0].set_title('Cumulative Execution Progress')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Execution price vs market price
axes[1].plot(market_data.index, market_data['price'], 
             linewidth=2, color='gray', alpha=0.5, label='Market Price')
axes[1].scatter(twap_executions['timestamp'], twap_executions['price'], 
                s=50, alpha=0.6, label='TWAP Executions', marker='o')
axes[1].scatter(vwap_executions['timestamp'], vwap_executions['price'], 
                s=30, alpha=0.6, label='VWAP Executions', marker='s')
axes[1].scatter(pov_executions['timestamp'], pov_executions['price'], 
                s=20, alpha=0.6, label='POV Executions', marker='^')
axes[1].set_ylabel('Price ($)')
axes[1].set_xlabel('Time')
axes[1].set_title('Execution Prices vs Market Price')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Performance Comparison

Compare the three algorithms across multiple metrics.

In [ ]:
# Create comparison DataFrame
comparison = pd.DataFrame({
    'Algorithm': ['TWAP', 'VWAP', 'POV'],
    'Avg Price': [
        twap_performance['avg_execution_price'],
        vwap_performance['avg_execution_price'],
        pov_performance['avg_execution_price']
    ],
    'Cost (bps)': [
        twap_performance['cost_vs_benchmark_bps'],
        vwap_performance['cost_vs_benchmark_bps'],
        pov_performance['cost_vs_benchmark_bps']
    ],
    'Num Executions': [
        twap_performance['num_executions'],
        vwap_performance['num_executions'],
        pov_performance['num_executions']
    ],
    'Price Std': [
        twap_performance['price_std'],
        vwap_performance['price_std'],
        pov_performance['price_std']
    ],
    'Quantity Std': [
        twap_performance['quantity_std'],
        vwap_performance['quantity_std'],
        pov_performance['quantity_std']
    ]
})

print('\nAlgorithm Performance Comparison:')
print('=' * 80)
print(comparison.to_string(index=False))
print('=' * 80)
print('\nKey Insights:')
print(f'- Arrival price: ${arrival_price:.2f}')
print(f'- Best average price: {comparison.loc[comparison["Avg Price"].idxmin(), "Algorithm"]}')
print(f'- Lowest cost: {comparison.loc[comparison["Cost (bps)"].idxmin(), "Algorithm"]}')
print(f'- Most executions: {comparison.loc[comparison["Num Executions"].idxmax(), "Algorithm"]}')

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Cost comparison
axes[0].bar(comparison['Algorithm'], comparison['Cost (bps)'], 
            color=['blue', 'green', 'red'], alpha=0.7)
axes[0].set_ylabel('Cost (basis points)')
axes[0].set_title('Execution Cost vs Arrival Price')
axes[0].grid(True, alpha=0.3, axis='y')

# Number of executions
axes[1].bar(comparison['Algorithm'], comparison['Num Executions'], 
            color=['blue', 'green', 'red'], alpha=0.7)
axes[1].set_ylabel('Number of Executions')
axes[1].set_title('Trade Frequency')
axes[1].grid(True, alpha=0.3, axis='y')

# Price variance
axes[2].bar(comparison['Algorithm'], comparison['Price Std'], 
            color=['blue', 'green', 'red'], alpha=0.7)
axes[2].set_ylabel('Price Std Dev ($)')
axes[2].set_title('Execution Price Variance')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Cost Analysis

Calculate total execution costs including market impact.

In [ ]:
# Calculate total costs in dollars
twap_total_cost = (twap_performance['avg_execution_price'] - arrival_price) * total_quantity
vwap_total_cost = (vwap_performance['avg_execution_price'] - arrival_price) * total_quantity
pov_total_cost = (pov_performance['avg_execution_price'] - arrival_price) * total_quantity

# Calculate total values
twap_total_value = twap_executions['value'].sum()
vwap_total_value = vwap_executions['value'].sum()
pov_total_value = pov_executions['value'].sum()

print('\nTotal Execution Costs:')
print('=' * 70)
print(f'Benchmark value (at arrival): ${arrival_price * total_quantity:,.2f}')
print()
print(f'TWAP:')
print(f'  Total value: ${twap_total_value:,.2f}')
print(f'  Total cost: ${twap_total_cost:,.2f}')
print(f'  Cost %: {twap_total_cost / (arrival_price * total_quantity) * 100:.3f}%')
print()
print(f'VWAP:')
print(f'  Total value: ${vwap_total_value:,.2f}')
print(f'  Total cost: ${vwap_total_cost:,.2f}')
print(f'  Cost %: {vwap_total_cost / (arrival_price * total_quantity) * 100:.3f}%')
print()
print(f'POV:')
print(f'  Total value: ${pov_total_value:,.2f}')
print(f'  Total cost: ${pov_total_cost:,.2f}')
print(f'  Cost %: {pov_total_cost / (arrival_price * total_quantity) * 100:.3f}%')
print('=' * 70)

## Conclusion

In this notebook, we've explored three fundamental execution algorithms:

### Algorithm Characteristics:

**TWAP (Time-Weighted Average Price)**
- Executes uniformly over time
- Simple and predictable
- Good for low-urgency orders
- May miss optimal volume windows

**VWAP (Volume-Weighted Average Price)**
- Follows historical volume patterns
- Minimizes market impact by trading when market is active
- Widely used as a benchmark
- Good for tracking market prices

**POV (Percentage of Volume)**
- Maintains consistent market participation
- Adapts to real-time volume changes
- Good for large orders requiring stealth
- Completion time is uncertain

### Key Takeaways:
- Choice of algorithm depends on order urgency, size, and market conditions
- Volume-aware strategies (VWAP, POV) typically reduce market impact
- Execution costs include both price movement and market impact
- Real-world implementation requires additional considerations like order book dynamics

### Next Steps:
- Explore feature engineering for predictive models (notebook 04)
- Analyze portfolio risk metrics (notebook 05)
- Study optimal execution theory (notebook 06)